### PromptTemplate 
* [PromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html#langchain_core.prompts.prompt.PromptTemplate)
* [ChatPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
* [ChatMessagePromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatMessagePromptTemplate.html#langchain_core.prompts.chat.ChatMessagePromptTemplate)
* [FewShotPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.few_shot.FewShotPromptTemplate.html#langchain_core.prompts.few_shot.FewShotPromptTemplate)
* PartialPrompt

In [28]:
# uv add python-dotenv langchain langchain-openai

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- `get_llm(temperature=0.7, provider="groq")`처럼 **`provider`를 지정해 필요할 때 LLM을 다시 생성**할 수 있습니다. (`"groq"` 또는 `"openai"`)
- `provider`를 생략하면 아래 설정 셀의 `PROVIDER` 값이 기본값으로 사용됩니다.
- 이 노트북의 LLM 생성 셀은 `provider="groq"`로 지정되어 있습니다. OpenAI를 쓰려면 해당 셀의 값을 `"openai"`로 바꾸세요. (OpenAI 사용 시 요금 발생)
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다.

In [4]:
from langchain_openai import ChatOpenAI

# 기본 LLM 제공자: "groq" 또는 "openai" — get_llm() 호출 시 provider 를 생략하면 이 값을 사용
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

##### 1) PromptTemplate 의 from_template() 함수 사용
* 주로 LLM(텍스트 완성형 모델, ex. Ollama, GPT-3.5)과 함께 사용
* 하나의 문자열 프롬프트를 생성

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

llm = get_llm(temperature=0.7, provider="groq")  # 제공자를 직접 지정 (키/모델명은 공통 설정 셀의 get_llm 사용)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [6]:

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"model_name":"ChatGPT", "count":3})
pprint(response)

('ChatGPT는 방대한 인터넷 텍스트를 토큰 단위로 나누어, 앞에 나온 토큰들을 보고 다음에 올 토큰을 예측하도록 스스로 학습합니다. 이 '
 '과정은 트랜스포머 구조의 여러 어텐션 레이어와 층을 이용해 파라미터를 조정하는 경사 하강법과 역전파 알고리즘으로 이루어집니다. 기본 사전 '
 '학습 후에는 인간 피드백을 활용한 강화학습(RLHF) 등으로 모델을 미세조정해 실제 대화에 맞는 응답을 생성하도록 최적화합니다.')


##### 2) PromptTemplate 결합하기
* 동일한 Prompt 패턴을 사용하지만 여러 개의 질문을 작성해서 LLM을 실행할 수도 있습니다.

In [7]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# 문자열 템플릿 결합 (PromptTemplate + PromptTemplate + 문자열)
combined_prompt = (
              prompt_template
              + PromptTemplate.from_template("\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요")
              + "\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요."
)
combined_prompt.format(model_name="ChatGPT", count=3, language="한국어")
print(combined_prompt)

chain = combined_prompt | llm | StrOutputParser()
response = chain.invoke({"model_name":"Gemini", "count":3, "language":"한국어"})

pprint(response)

input_variables=['count', 'language', 'model_name'] input_types={} partial_variables={} template='{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요.\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요.'
('**Gemini 모델의 학습 원리 (3문장)**  \n'
 '1. Gemini는 대규모 텍스트와 멀티모달(이미지·텍스트) 데이터를 수집해, 자기지도 학습(self‑supervised '
 'learning)과 지도 학습(supervised fine‑tuning)을 병행합니다.  \n'
 '2. 사전 학습 단계에서는 트랜스포머 기반의 디코더와 인코더를 동시에 훈련시켜, 입력 시퀀스의 다음 토큰을 예측하도록 하여 언어와 시각 '
 '정보를 통합적으로 이해하도록 합니다.  \n'
 '3. 이후 인간 피드백을 활용한 강화 학습(RLHF) 과정을 거쳐, 사용자의 의도에 맞는 고품질 응답과 안전성을 확보합니다.  \n'
 '\n'
 '**Gemini 모델의 주요 장점**  \n'
 '- 텍스트와 이미지를 동시에 처리할 수 있는 멀티모달 능력이 뛰어나, 복합적인 질의에 대해 풍부하고 정확한 답변을 제공한다.  \n'
 '- 대규모 사전 학습과 RLHF를 결합해 높은 언어 이해도와 생성 품질을 유지하면서도, 안전하고 편향을 최소화한 응답을 생성한다.  \n'
 '- 구글 클라우드와 연동된 인프라를 활용해 실시간 확장성과 빠른 응답 속도를 제공한다.  \n'
 '\n'
 '**Gemini와 비슷한 AI 모델 (한국어 명칭)**  \n'
 '- 챗GPT (ChatGPT)  \n'
 '- 라마 (LLaMA)  \n'
 '- 클로드 (Claude)  \n'
 '- 얼니봇 (Ernie Bot)  \n'
 '- 미스트랄 (Mi

#### PromptTemplate 의 파라미터를 배열 형태로 하여 여러개 사용하는 경우

In [ ]:
# List Comprehension 예제
# before (적용하지 않은 경우)
result_list = []
for val in range(10):
    # 2의
    if val % 2 == 0:
        result_list.append(val ** 2)

result_list        

In [ ]:
# after (적용한 경우)
result_list2 = [val**2 for val in range(10) if val % 2 == 0]
result_list2

In [8]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

questions = [
    {"model_name": "GPT-4", "count": 3}, #dict
    {"model_name": "Gemini", "count": 4},
    {"model_name": "Claude", "count": 4},
]

# 여러 개의 프롬프트를 미리 생성
# q변수가 dict 타입
formatted_prompts = [prompt_template.format(**q) for q in questions]
print(formatted_prompts)  # 미리 생성된 질문 목록 확인

for prompt in formatted_prompts:
    print(type(prompt), prompt)
    response = llm.invoke(prompt)
    pprint(response.content)
    print("-" * 50)

['GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.', 'Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.', 'Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.']
<class 'str'> GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.
('GPT-4는 대규모 텍스트 데이터를 사전 학습(pre‑training) 단계에서 입력‑출력 쌍으로 변환해, 다음에 올 단어를 예측하도록 '
 '신경망을 훈련합니다. 이때 트랜스포머 아키텍처를 사용해 문맥을 양방향으로 파악하고, 어텐션 메커니즘으로 중요한 단어에 가중치를 '
 '부여합니다. 이후 특정 작업에 맞게 소량의 라벨 데이터로 미세 조정(fine‑tuning)하거나 인간 피드백을 활용한 '
 '강화학습(RLHF)으로 성능을 최적화합니다.')
--------------------------------------------------
<class 'str'> Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Gemini 모델은 방대한 텍스트 데이터를 이용해 트랜스포머 기반의 신경망을 사전 학습합니다.  \n'
 '학습 과정에서는 입력 문장의 일부를 마스킹하거나 다음 토큰을 예측하도록 하여, 자기지도(self‑supervised) 방식으로 언어 '
 '패턴을 스스로 학습합니다.  \n'
 '사전 학습이 끝난 뒤에는 인간 피드백을 활용한 강화학습(RLHF) 등을 통해 모델의 응답 품질과 안전성을 향상시킵니다.  \n'
 '이러한 단계적 학습 흐름을 통해 Gemini는 다양한 언어 이해와 생성 능력을 갖춘 다목적 AI 모델이 됩니다.')
--------------------------------------------------
<class 'str'> Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Claude 모델은 대규모 텍스트 데이터를 이용해 사전 학습(pre‑training)을 진행합니다. 이

##### 2) ChatPromptTemplate
* Tuple 형태의 system, user, assistant 메시지 지원
* 여러 개의 메시지를 조합하여 LLM에게 전달 가능
* 간결성과 가독성이 높고 단순한 구조

In [18]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    # role, message
    ("system", "This system is an expert in answering questions about {topic}. \
     Please provide clear and detailed explanations."),
    ("user", "{model_name} 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요? 이 모델을 개발한 회사는 어디인가요?"),
])

messages = chat_prompt.format_messages(topic="AI", model_name="ExaOne")
print(messages)

# 생성한 메시지를 바로 주입하여 호출하기
llm = get_llm(temperature=0.7, provider="groq") 
response = llm.invoke(messages)

print(type(response))
print(response.content)

[SystemMessage(content='This system is an expert in answering questions about AI.      Please provide clear and detailed explanations.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ExaOne 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요? 이 모델을 개발한 회사는 어디인가요?', additional_kwargs={}, response_metadata={})]
[LLM 생성] provider=groq | model=openai/gpt-oss-120b
<class 'langchain_core.messages.ai.AIMessage'>
### ExaOne 모델에 대한 현재 알려진 정보

2024년 6월까지 제가 학습한 공개 자료에는 **“ExaOne”**이라는 이름의 AI 모델에 대한 구체적인 정보가 포함되어 있지 않습니다. 따라서 아래 질문에 대해 확정적인 답변을 드리기 어렵습니다.

| 질문 | 현재 알려진 내용 |
|------|-------------------|
| **학습 원리** | 공개된 자료가 없어 정확히 알 수 없습니다. |
| **개발 국가** | 공개된 자료가 없어 알 수 없습니다. |
| **개발 회사** | 공개된 자료가 없어 알 수 없습니다. |

---

## 그래도 도움이 될 수 있는 일반적인 내용

만약 “ExaOne”이 **대규모 언어 모델(LLM)** 혹은 **멀티모달 모델**이라면, 현재(2024년) 대부분의 최신 AI 모델이 사용하는 **학습 원리**와 **개발 흐름**은 다음과 같습니다.

### 1. 데이터 수집 및 전처리
1. **대규모 텍스트/멀티모달 데이터**  
   - 웹 크롤링, 책, 논문, 뉴스, 포럼, 코드 저장소 등 다양한 출처에서 수십억~수조 토큰을 수집합니다.  
   - 멀티모달 모델이라면

In [19]:
# 체인을 생성하여 호출하기
chain = chat_prompt | llm | StrOutputParser()

response = chain.invoke({"topic":"AI", "model_name":"deepseek"})
print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
## DeepSeek 모델 개요  
- **모델명**: DeepSeek‑Chat, DeepSeek‑Coder, DeepSeek‑MoE 등 여러 변종이 존재합니다.  
- **개발 국가**: **중국** (베이징)  
- **개발 기업**: **DeepSeek** (딥시크) – 2023년에 설립된 AI 스타트업으로, 전 OpenAI·Google·Microsoft 연구원들이 공동 창업했습니다.  

---

## 1. 학습 원리 (Training Principle)

### 1) 기본 아키텍처 – Decoder‑only Transformer
- **구조**: GPT‑계열과 동일한 **Decoder‑only Transformer**.  
- **핵심 요소**  
  - **Multi‑Head Self‑Attention**: 입력 토큰 간의 전역적인 관계를 학습.  
  - **Feed‑Forward Network (FFN)**: 각 토큰에 비선형 변환 적용.  
  - **Layer Normalization & Residual Connection**: 학습 안정화와 깊은 네트워크 전파를 돕습니다.  

### 2) 사전 학습 (Pre‑training)

| 단계 | 설명 |
|------|------|
| **데이터 수집** | 웹 크롤링, 위키피디아, 책, 논문, 코드 저장소(GitHub) 등 다양한 공개·비공개 텍스트·코드 데이터를 수집. 총 **수백 테라바이트** 규모(예: 1.2 trillion 토큰) |
| **토크나이징** | Byte‑Pair Encoding (BPE) 혹은 **SentencePiece** 기반 **Tokenizer**를 사용해 32 k~64 k 정도의 서브워드 vocab을 구축. |
| **목표 함수** | **다음 토큰 예측 (Causal Language Modeling)**: \(\mathcal{L} = -\sum_{t} \log P(x_t

#### 3) ChatPromptTemplate
* SystemMessagePromptTemplate와 HumanMessagePromptTemplate 클래스 사용
* 객체 지향적 접근 - Message 객체를 독립적으로 생성 가능
* 여러 조건에 따라 다른 시스템 메시지 선택

```python
if user_is_beginner:
    system_message = SystemMessagePromptTemplate.from_template("초보자를 위한 설명: {topic}")
else:
    system_message = SystemMessagePromptTemplate.from_template("전문가를 위한 상세 분석: {topic}")
```

In [ ]:
# ChatMessagePromptTemplate 활용

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    ChatMessagePromptTemplate
)
from langchain_openai import ChatOpenAI

# 개별 메시지 템플릿 정의
system_message = SystemMessagePromptTemplate.from_template(
    "You are an AI expert in {topic}. Please provide clear and detailed explanations."
)
user_message = HumanMessagePromptTemplate.from_template(
    "{question}"
)
ai_message = AIMessagePromptTemplate.from_template(
    "This is an example answer about {topic}."
)

# ChatPromptTemplate로 메시지들을 묶기
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    user_message,
    ai_message
])

# 메시지 생성
messages = chat_prompt.format_messages(topic="건강", question="독감예방 접종을 꼭 맞아야 하나요?")

# LLM 호출 (공통 설정 셀의 llm 사용)
response = llm.invoke(messages)

# 결과 출력
print(response.content)

#### ChatMessagePromptTemplate는 여러 종류의 메시지(시스템, 인간, AI)를 조합하여 복잡한 프롬프트를 생성할 때 유용합니다.
* SystemMessagePromptTemplate: 이 템플릿은 AI 모델에게 역할을 부여하거나 전반적인 규칙을 설정하는 시스템 메시지를 만듭니다. 위의 예시에서는 "번역을 도와주는 유용한 도우미"라는 역할을 지정합니다.
* HumanMessagePromptTemplate: 이 템플릿은 사용자의 질문이나 요청을 담는 인간 메시지를 만듭니다. 아래의 예시에서는 번역할 텍스트를 입력받습니다.
* ChatPromptTemplate.from_messages: 이 클래스 메서드는 시스템 메시지, 인간 메시지 등 여러 종류의 MessagePromptTemplate 객체들을 리스트로 받아 하나의 채팅 프롬프트 템플릿으로 통합합니다.
* format_messages: 이 메서드는 정의된 템플릿에 실제 값을 채워 넣어 [SystemMessage, HumanMessage] 형태의 리스트를 반환합니다. 이 리스트는 채팅 모델(Chat Model) 에 바로 전달될 수 있습니다.

In [ ]:
# 필요한 라이브러리 임포트
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 1. SystemMessagePromptTemplate와 HumanMessagePromptTemplate 생성
# SystemMessagePromptTemplate는 모델의 페르소나 또는 기본 지침을 설정합니다.
system_template = "You are a helpful assistant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# HumanMessagePromptTemplate는 사용자로부터 받는 입력 프롬프트를 정의합니다.
human_template = "{text_to_translate}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# 2. ChatPromptTemplate 생성
# 위에서 만든 두 템플릿을 리스트로 묶어 ChatPromptTemplate을 만듭니다.
chat_prompt_template = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# 3. 프롬프트 포맷팅
# chat_prompt_template.format_messages()를 사용하여 최종 메시지 리스트를 생성합니다.
# 이 함수는 딕셔너리 형태의 입력 변수를 받습니다.
formatted_prompt = chat_prompt_template.format_messages(
    input_language="English",
    output_language="Korean",
    text_to_translate="Chaining can mean making multiple LLM calls in a sequence. Language models are often non deterministic and can make errors, so making multiple calls to check previous outputs or to break down larger tasks into bite-sized steps can improve results."
)

# 4. 결과 출력
print(formatted_prompt)

# LLM 호출
response = llm.invoke(formatted_prompt)

# 결과 출력
print(response.content)

##### 4) FewShotPromptTemplate
* FewShotPromptTemplate은 모델이 특정 형식을 따르게 하거나, 일관된 응답을 생성하도록 유도할 때 유용합니다.
* 도메인 지식이 필요하거나, AI가 오답을 줄이고 더 신뢰할 만한 답변을 생성하도록 해야 할 때 효과적입니다.

##### 4-1) PromptTemplate을 사용하지 않는 경우

In [21]:
# PromptTemplate을 사용하지 않는 경우

# model
print(llm.model_name)

# chain 실행
result = llm.invoke("태양계의 행성들을 간략히 정리해 주세요.")

print(type(result))
print(result.content)

openai/gpt-oss-120b
<class 'langchain_core.messages.ai.AIMessage'>
### 태양계 행성 간략 정리

| 순서 (태양으로부터) | 행성 | 종류 | 주요 특징 |
|-------------------|------|------|-----------|
| 1 | **수성 (Mercury)** | 암석행성 | 가장 작은·가장 뜨거운 낮(≈ 430 °C)·극단적인 온도 차(밤≈ ‑180 °C) |
| 2 | **금성 (Venus)** | 암석행성 | 가장 두꺼운 대기(이산화탄소)·표면압 92배·극강 온실효과(≈ 470 °C) |
| 3 | **지구 (Earth)** | 암석행성 | 물이 액체 상태로 존재·생명체가 사는 유일한 행성·대기: N₂·O₂ |
| 4 | **화성 (Mars)** | 암석행성 | “붉은 행성”·극지에 물 얼음·대기 얇고 CO₂ 주성분·과거 물 흐름 흔적 |
| 5 | **목성 (Jupiter)** | 가스행성 | 가장 큰 행성·주성분은 수소·헬륨·대형 폭풍(대적점)·79개의 위성 |
| 6 | **토성 (Saturn)** | 가스행성 | 눈부신 고리 시스템(얼음·암석)·주성분은 수소·헬륨·82개의 위성 |
| 7 | **천왕성 (Uranus)** | 얼음거인 | 자전축이 98° 기울어 “옆으로 누운” 행성·대기: 메탄·수소·헬륨·27개의 위성 |
| 8 | **해왕성 (Neptune)** | 얼음거인 | 가장 바람이 강한 행성(최대 2,100 km/h)·청색 대기(메탄)·14개의 위성 |

**핵심 포인트**  
- **암석행성**(수성·금성·지구·화성)은 고체 표면을 가지고 있고, 크기가 비교적 작다.  
- **가스·얼음거인**(목성·토성·천왕성·해왕성)은 주로 수소·헬륨(가스) 또는 물·암모니아·메탄(얼음)으로 이루어져 있다.  
- 행성마다 대기 구성, 온도, 위성·고리 시스템 등이 크게 다르며, 이는 각 행성의 형성과 진화 과정에 큰 영향을 미친다.


##### 4-2) FewShotChatMessagePromptTemplate 사용하는 경우

In [22]:
# FewShotChatMessagePromptTemplate 사용하는 경우
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 예시는 시스템 프롬프트(초등학생 눈높이)와 같은 톤·같은 형식(제목 + 비유 글머리표 + 한 줄 정리)으로 통일
examples_list = [
    {
        "input": "뉴턴의 운동 법칙을 요약해 주세요.",
        "output": """### 뉴턴의 운동 법칙
- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.
- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.
- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.
 한 줄 정리: 힘은 항상 짝으로 움직여요!"""
    },
    {
        "input": "지구의 대기 구성 요소를 알려주세요.",
        "output": """### 지구 대기의 구성
- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.
- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.
- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.
 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!"""
    }
]

# 예제 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# FewShotChatMessagePromptTemplate 적용
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples_list,
)

# 최종 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# 모델 생성 및 체인 구성
chain = final_prompt | llm
print(llm.model_name)
print(chain)

# 테스트 실행
result = chain.invoke({"input": "전기의 발생 원리에 대하여 설명해 줘"})
print(result.content)

openai/gpt-oss-120b
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': '뉴턴의 운동 법칙을 요약해 주세요.', 'output': '### 뉴턴의 운동 법칙\n- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.\n- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.\n- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.\n 한 줄 정리: 힘은 항상 짝으로 움직여요!'}, {'input': '지구의 대기 구성 요소를 알려주세요.', 'output': '### 지구 대기의 구성\n- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.\n- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.\n- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.\n 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptT

#### 5-1) PartialPrompt 
* 프롬프트를 더 동적으로 활용할 수 있으며, AI 응답을 더 일관성 있게 조정 가능함

In [27]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# 프롬프트 템플릿 정의 (부분 변수 적용)
prompt = PromptTemplate(
    template="{season}에 일어나는 대표적인 지구과학 현상은 {phenomenon}이 맞나요? \
        {season}에 주로 발생하는 지구과학 현상을 3개 알려주세요",
    input_variables=["phenomenon"],  # 사용자 입력 필요
    # 함수 자체를 전달 → format() 호출 시점마다 현재 계절을 계산
    partial_variables={"season": get_current_season}  # 동적으로 계절 값 할당
)


# 특정 계절의 현상 질의
query = prompt.format(phenomenon="태풍 발생")
llm = get_llm(temperature=0.7, provider="groq")  # 제공자를 직접 지정 (키/모델명은 공통 설정 셀의 get_llm 사용)
result = llm.invoke(query)


# 결과 출력
print(f" 프롬프트: {query}")
print(f" 모델 응답: {result.content}")

[LLM 생성] provider=groq | model=openai/gpt-oss-120b
 프롬프트: 가을에 일어나는 대표적인 지구과학 현상은 태풍 발생이 맞나요?         가을에 주로 발생하는 지구과학 현상을 3개 알려주세요
 모델 응답: ### 가을에 자주 나타나는 대표적인 지구과학 현상  

| 번호 | 현상 | 간단한 설명 | 왜 가을에 주로 나타나는가 |
|------|------|------------|---------------------------|
| 1 | **중위도 저기압·전선 활동·강수** | 여름에 활발하던 열대 저기압(태풍) 대신, 북반구 중위도(30°–60°)에서 형성되는 저기압과 한랭·온난 전선이 자주 교차하면서 비·눈·소나기가 많이 발생한다. | 가을은 남쪽에서 북쪽으로 이동하는 따뜻한 공기와, 북쪽에서 내려오는 차가운 대륙성 공기가 겹치는 계절이어서 전선이 활발해진다. |
| 2 | **대기 온도 역전·안개·연무** | 지표면이 급격히 냉각되면서 지표면 근처의 공기가 위층보다 차가워지는 ‘역전층’이 형성된다. 이때 수증기가 응결해 안개·연무가 자주 발생한다. | 일교차가 크게 벌어지는 가을 저녁·새벽에 지표면이 빠르게 식고, 습한 공기가 위에 머무르기 쉬워 역전 현상이 빈번해진다. |
| 3 | **해양·대기 계절 전환에 따른 해류·바람 변화** | 동해·남해·서해 등 주변 해역에서는 가을에 해수면 온도가 급격히 낮아지면서 ‘가을 전환 해류’가 형성되고, 이에 따라 동풍·남풍·북풍이 교차한다. 이때 해상풍·해무·해안 지역의 급격한 온도·습도 변화가 일어난다. | 태양 복사량이 감소하면서 해수면이 서서히 식고, 대기와 해양 사이의 열·수분 교환 패턴이 바뀌어 해류와 바람이 전환된다. |

---

## 1️⃣ 태풍은 가을에도 발생하지만 “대표적인 가을 현상”인가?

- **태풍 시즌**은 대체로 **6월 ~ 10월**이며, 특히 **7~9월**에 가장 활발합니다.  
- **가을(9~11월)** 초반까지는 아직 따뜻한 남해

In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# Step 1: 현재 계절 결정
season_name = get_current_season()
#season_name = get_current_season("south")  # 계절 값 얻기
print(f"현재 계절: {season_name}")

In [ ]:

# Step 2: 해당 계절의 자연 현상 추천
prompt2 = ChatPromptTemplate.from_template(
    "{season}에 주로 발생하는 대표적인 지구과학 현상 3가지를 알려주세요. "
    "각 현상에 대해 간단한 설명을 포함해주세요."
)

print(llm.model_name)
# 체인 2: 자연 현상 추천 (입력: 계절 → 출력: 자연 현상 목록)
chain2 = (
    {"season": lambda x : season_name}  # chain1의 출력을 season 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: 현재 계절에 따른 자연 현상 추천
response = chain2.invoke({})
print(f"\n {season_name}에 발생하는 자연 현상:\n{response}")

#### 5-2) PartialPrompt 
* API 호출 데이터, 시간 정보, 사용자 정보 등을 반영할 때 매우 유용함

In [ ]:
import requests
from datetime import datetime
from langchain_core.prompts import PromptTemplate

EXCHANGE_API_URL = "https://api.exchangerate-api.com/v4/latest/USD"


def get_exchange_rate() -> str:
    """USD→KRW 환율과 API 기준 일자를 문자열로 반환합니다."""
    response = requests.get(EXCHANGE_API_URL, timeout=10)
    response.raise_for_status()
    data = response.json()
    base_date = data.get("date", datetime.now().strftime("%Y-%m-%d"))
    return f"{base_date} 기준 1달러 = {data['rates']['KRW']}원"


# Partial Prompt 활용 (함수 자체를 전달 → format() 호출 시점마다 실행)
prompt = PromptTemplate(
    template=(
        "오늘 날짜: {today}\n"
        "환율 정보: {info}\n\n"
        "위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. "
        "제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요."
    ),
    input_variables=[],  # 사용자 입력 없음
    partial_variables={
        "today": lambda: datetime.now().strftime("%Y-%m-%d"),
        "info": get_exchange_rate,  # API에서 가져온 데이터 자동 반영
    },
)

# LLM 모델명
print(llm.model_name)

# 프롬프트를 한 번만 생성해 재사용 (API 중복 호출 방지)
formatted = prompt.format()

# 모델에 프롬프트 전달 및 응답 받기
response = llm.invoke(formatted)

# 결과 출력
print(" 프롬프트:", formatted)
print(" 모델 응답:", response.content)